<a href="https://colab.research.google.com/github/shishiradk/0-to-hero/blob/main/micrograd_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# micrograd exercises

1. watch the [micrograd video](https://www.youtube.com/watch?v=VMj-3S1tku0) on YouTube
2. come back and complete these exercises to level up :)

## section 1: derivatives

In [1]:
# here is a mathematical expression that takes 3 inputs and produces one output
from math import sin, cos

def f(a, b, c):
  return -a**3 + sin(3*b) - 1.0/c + b**2.5 - a**0.5

print(f(2, 3, 4))

6.336362190988558


In [2]:
# write the function df that returns the analytical gradient of f
# i.e. use your skills from calculus to take the derivative, then implement the formula
# if you do not calculus then feel free to ask wolframalpha, e.g.:
# https://www.wolframalpha.com/input?i=d%2Fda%28sin%283*a%29%29%29

def gradf(a, b, c):
  df_da = -3*a**2 - 0.5*a**(-0.5)
  df_db = 3*cos(3*b) + 2.5*b**1.5
  df_dc = c**(-2)
  return [df_da, df_db,df_dc]
  return [0, 0, 0] # todo, return [df/da, df/db, df/dc]

# expected answer is the list of
ans = [-12.353553390593273, 10.25699027111255, 0.0625]
yours = gradf(2, 3, 4)
for dim in range(3):
  ok = 'OK' if abs(yours[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {yours[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353553390593273
OK for dim 1: expected 10.25699027111255, yours returns 10.25699027111255
OK for dim 2: expected 0.0625, yours returns 0.0625


In [4]:
# now estimate the gradient numerically without any calculus, using
# the approximation we used in the video.
# you should not call the function df from the last cell
h = 0.0001
a,b,c = 2,3,4
base_val = f(a,b,c)
# Nudge 'a'
f_a = f(a+h,b,c)
numerical_grad_a = (f_a - base_val)/h

#Nudge "b"
f_b = f(a,b+h,c)
numerical_grad_b = (f_b - base_val)/h

# Nudge "c"
f_c = f(a,b,c+h)
numerical_grad_c = (f_c - base_val)/h


numerical_grad = [numerical_grad_a, numerical_grad_b, numerical_grad_c] # TODO
# -----------

for dim in range(3):
  ok = 'OK' if abs(numerical_grad[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad[dim]}")

WRONG! for dim 0: expected -12.353553390593273, yours returns -12.354148981312818
WRONG! for dim 1: expected 10.25699027111255, yours returns 10.25712962014147
OK for dim 2: expected 0.0625, yours returns 0.06249843753636242


In [7]:
# there is an alternative formula that provides a much better numerical
# approximation to the derivative of a function.
# learn about it here: https://en.wikipedia.org/wiki/Symmetric_derivative
# implement it. confirm that for the same step size h this version gives a
# better approximation.
h = 0.0001
a,b,c = 2,3,4

# Symmetric derivative for 'a' (note the parentheses around 2 * h)
numerical_grad2_a = (f(a + h, b, c) - f(a - h, b, c)) / (2 * h)

# Symmetric derivative for 'b' (keep parameter order as (a, b, c))
numerical_grad2_b = (f(a, b + h, c) - f(a, b - h, c)) / (2 * h)

# Symmetric derivative for 'c' (keep parameter order as (a, b, c))
numerical_grad2_c = (f(a, b, c + h) - f(a, b, c - h)) / (2 * h)

numerical_grad2 = [numerical_grad2_a, numerical_grad2_b, numerical_grad2_c]

for dim in range(3):
  ok = 'OK' if abs(numerical_grad2[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad2[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353553400714645
OK for dim 1: expected 10.25699027111255, yours returns 10.25699031393934
OK for dim 2: expected 0.0625, yours returns 0.06250000003760192


## section 2: support for softmax

In [14]:
class Value:
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"

  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')

    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward

    return out

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')

    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward

    return out

  def __pow__(self, other):
    assert isinstance(other, (int, float)), "only supporting int/float powers for now"
    out = Value(self.data ** other, (self,), f'**{other}')

    def _backward():
      self.grad += other * (self.data ** (other - 1)) * out.grad
    out._backward = _backward

    return out

  def __truediv__(self, other):
    return self * (other ** -1)

  def __neg__(self):
    return self * -1

  def __sub__(self, other):
    return self + (-other)

  def log(self):
    out = Value(log(self.data), (self,), 'log')

    def _backward():
      self.grad += (1.0 / self.data) * out.grad
    out._backward = _backward

    return out

  def exp(self):
    out = Value(exp(self.data), (self,), 'exp')

    def _backward():
      self.grad += out.data * out.grad
    out._backward = _backward

    return out

  def backward(self):
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)

    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

In [16]:
# without referencing our code/video __too__ much, make this cell work
# you'll have to implement (in some cases re-implemented) a number of functions
# of the Value object, similar to what we've seen in the video.
# instead of the squared error loss this implements the negative log likelihood
# loss, which is very often used in classification.

# this is the softmax function
# https://en.wikipedia.org/wiki/Softmax_function
def softmax(logits):
  counts = [logit.exp() for logit in logits]
  denominator = sum(counts, Value(0.0))
  out = [c / denominator for c in counts]
  return out

# this is the negative log likelihood loss function, pervasive in classification
logits = [Value(0.0), Value(3.0), Value(-2.0), Value(1.0)]
probs = softmax(logits)
loss = -probs[3].log() # dim 3 acts as the label for this input example
loss.backward()
print(loss.data)

ans = [0.041772570515350445, 0.8390245074625319, 0.005653302662216329, -0.8864503806400986]
for dim in range(4):
  ok = 'OK' if abs(logits[dim].grad - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {logits[dim].grad}")


2.1755153626167147
OK for dim 0: expected 0.041772570515350445, yours returns 0.041772570515350445
OK for dim 1: expected 0.8390245074625319, yours returns 0.8390245074625319
OK for dim 2: expected 0.005653302662216329, yours returns 0.005653302662216329
OK for dim 3: expected -0.8864503806400986, yours returns -0.8864503806400986


In [18]:
# verify the gradient using the torch library
# torch should give you the exact same gradient
import torch

# Convert micrograd Value objects to torch Tensors
logits_data = [l.data for l in logits]
logits_t = torch.tensor(logits_data, requires_grad=True)

# Apply softmax
probs_t = torch.softmax(logits_t, dim=0)

# Calculate loss, assuming the label is 3 (0-indexed) as in the micrograd example
loss_t = -torch.log(probs_t[3])

loss_t.backward()

print("pytorch gradients:", logits_t.grad.tolist())

pytorch gradients: [0.041772566735744476, 0.8390244841575623, 0.005653302650898695, -0.8864504098892212]
